# Feature Selection

This notebook will provide the justifcations for feature selection.

# Data Processing

In [1]:
# Packages 
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

In [2]:
# Load data
df = pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [3]:
# First 5 rows of dataset
df.show(5)

time,src_user,dest_user,src_comp,dest_comp,auth_type,logon_type,auth_orientation,outcome
i64,str,str,str,str,str,str,str,str
1,"""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C1250""","""C586""","""NTLM""","""Network""","""LogOn""","""Success"""
1,"""ANONYMOUS LOGON@C586""","""ANONYMOUS LOGON@C586""","""C586""","""C586""","""?""","""Network""","""LogOff""","""Success"""
1,"""C101$@DOM1""","""C101$@DOM1""","""C988""","""C988""","""?""","""Network""","""LogOff""","""Success"""
1,"""C1020$@DOM1""","""SYSTEM@C1020""","""C1020""","""C1020""","""Negotiate""","""Service""","""LogOn""","""Success"""
1,"""C1021$@DOM1""","""C1021$@DOM1""","""C1021""","""C625""","""Kerberos""","""Network""","""LogOn""","""Success"""


In [4]:
# Keep only human users
df = df.filter(pl.col("src_user").str.starts_with("U"))

In [5]:
# Features dataframe
features = (
    df
    .with_columns(
        bucket=pl.col("time") // 3600,
        hour_of_day=(pl.col("time") // 3600) % 24,
        is_failure=(pl.col("outcome") != "Success").cast(pl.Int8),
    )
    .group_by(["src_user", "bucket"])
    .agg(
        hour_of_day=pl.col("hour_of_day").first(),
        n_events=pl.len(),
        n_failures=pl.col("is_failure").sum(),
        failure_ratio=pl.col("is_failure").mean(),
        n_distinct_dest=pl.col("dest_comp").n_unique(),
        n_distinct_src=pl.col("src_comp").n_unique(),
    )
    .collect()
)

In [6]:
features = features.with_columns(
    hod_sin=(2 * np.pi * pl.col("hour_of_day") / 24).sin(),
    hod_cos=(2 * np.pi * pl.col("hour_of_day") / 24).cos(),
)

In [7]:
features.show(5)

src_user,bucket,hour_of_day,n_events,n_failures,failure_ratio,n_distinct_dest,n_distinct_src,hod_sin,hod_cos
str,i64,i64,u64,i64,f64,u64,u64,f64,f64
"""U8385@DOM1""",837,21,7,0,0.0,4,4,-0.707107,0.707107
"""U1394@DOM1""",1373,5,14,0,0.0,3,4,0.965926,0.258819
"""U4510@DOM1""",374,14,4,0,0.0,1,2,-0.5,-0.866025
"""U4764@DOM1""",140,20,4,0,0.0,1,2,-0.866025,0.5
"""U1922@DOM1""",374,14,14,0,0.0,2,4,-0.5,-0.866025


# Exploratory Data Analysis

# Clustering 

In [8]:
# Packages
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [9]:
# Extract relevant features
feature_cols = [
    "n_events",
    "failure_ratio",
    "n_distinct_dest",
    "n_distinct_src",
    "hod_sin",
    "hod_cos",
]

X = features.select(feature_cols).to_numpy()

In [10]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [14]:
km = KMeans(n_clusters=4, n_init=10, random_state=42)
labels = km.fit_predict(X_scaled)

print(f"Cluster sizes: {np.bincount(labels)}")

Cluster sizes: [2614189  373997 1590552 2809451]


In [15]:
features_clustered = features.with_columns(cluster=pl.Series(labels))

(
    features_clustered
    .group_by("cluster")
    .agg(
        size=pl.len(),
        n_events_mean=pl.col("n_events").mean(),
        n_events_median=pl.col("n_events").median(),
        failure_ratio_mean=pl.col("failure_ratio").mean(),
        n_distinct_dest_mean=pl.col("n_distinct_dest").mean(),
        n_distinct_src_mean=pl.col("n_distinct_src").mean(),
        hour_of_day_mean=pl.col("hour_of_day").mean(),
    )
    .sort("cluster")
)

cluster,size,n_events_mean,n_events_median,failure_ratio_mean,n_distinct_dest_mean,n_distinct_src_mean,hour_of_day_mean
i32,u64,f64,f64,f64,f64,f64,f64
0,2614189,29.13559,17.0,0.000846,3.64142,3.794723,6.36128
1,373997,23.573729,4.0,0.992193,2.044626,1.997644,11.425153
2,1590552,110.34869,52.0,0.001555,9.809164,7.763822,11.145558
3,2809451,28.900547,17.0,0.000997,3.616919,3.71626,16.810969


In [16]:
X_log = (
    features
    .with_columns(
        pl.col("n_events").log1p(),
        pl.col("n_distinct_dest").log1p(),
        pl.col("n_distinct_src").log1p(),
    )
    .select(feature_cols)
    .to_numpy()
)

scaler_log = StandardScaler()
X_log_scaled = scaler_log.fit_transform(X_log)

# Confirm the scale problem is gone
print("Max per feature (log):", X_log_scaled.max(axis=0).round(1))
print("Min per feature (log):", X_log_scaled.min(axis=0).round(1))

# %%
km_log = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_log = km_log.fit_predict(X_log_scaled)

print(f"Cluster sizes (log): {np.bincount(labels_log)}")

# %%
features_clustered_log = features.with_columns(cluster=pl.Series(labels_log))

(
    features_clustered_log
    .group_by("cluster")
    .agg(
        size=pl.len(),
        n_events_mean=pl.col("n_events").mean(),
        n_events_median=pl.col("n_events").median(),
        failure_ratio_mean=pl.col("failure_ratio").mean(),
        n_distinct_dest_mean=pl.col("n_distinct_dest").mean(),
        n_distinct_src_mean=pl.col("n_distinct_src").mean(),
        hour_of_day_mean=pl.col("hour_of_day").mean(),
    )
    .sort("cluster")
)

Max per feature (log): [ 6.4  4.3 11.7 12.1  1.4  1.6]
Min per feature (log): [-2.  -0.2 -1.5 -1.9 -1.4 -1.2]
Cluster sizes (log): [2732853 2152776 2128492  374068]


cluster,size,n_events_mean,n_events_median,failure_ratio_mean,n_distinct_dest_mean,n_distinct_src_mean,hour_of_day_mean
i32,u64,f64,f64,f64,f64,f64,f64
0,2732853,82.584331,37.0,0.001404,7.631235,6.288877,11.403121
1,2152776,41.906277,22.0,0.000401,4.950123,5.14698,11.780627
2,2128492,8.177389,4.0,0.001394,1.773745,2.089652,11.773973
3,374068,22.405742,4.0,0.991512,2.033411,1.983508,11.42796


# References 

https://scikit-learn.org/stable/auto_examples/applications/plot_cyclical_feature_engineering.html

https://scikit-learn.org/stable/modules/clustering.html